# Mean-HRTF baselines — CARLab / HUTUBS / SONICOM

The single-database run recorded a CARLab validation baseline of 5.053 dB, but the
HUTUBS and SONICOM columns of the results tables were left blank, and **no test-set
baseline was ever computed** — `RUN_TEST` in the training notebook was never set to
`True`, and the locked metrics file comes from the figure cell, which does not compute
baselines. This notebook fills both gaps.

**What it does.** Loads the frozen split `split_v21.json`, verifies each database's
subject-list hash still matches, and computes the mean-HRTF baseline for every database
on both the validation and the test partition.

**What it does not do.** It never loads the model, never touches the GPU, and never
writes to `split_v21.json`. It is a pure statistic of the data, so it cannot change any
reported model number. Colab's free CPU runtime is enough; expect a few minutes,
dominated by unzipping.

---

### The convention, stated explicitly

The baseline answers: *how well would you do by predicting the training-set mean spectrum
for everybody?* Three decisions define it, and all three must match how the model is
scored, or the "72.8% reduction" claim is not a ratio of comparable quantities.

| Decision | What the training notebook does | Why |
|---|---|---|
| Whose mean? | **Train ears only** | A mean that has seen the val/test subjects is not a prior — it is a leak. |
| Pooled how? | **One RMSE over (all ears x 93 bins x 3072 directions)** | Identical to `evaluate()`, which is how 1.372 dB is produced. The arithmetic mean of per-ear RMSEs is a *different, systematically smaller* number. |
| Which bins? | **All 93** (`band_lo_hz = 0`, DC to 15.85 kHz) | Same band mask as the loss. |
| Scored on which ears? | **The same partition as the model figure it is compared with** | Val baseline pairs with val RMSE, test with test. |

The last row is the one that gets misremembered, because it is easy to conflate *whose
data the mean is built from* with *whose data the error is measured on*. They are separate
choices, and the answers differ:

- The mean is built from **train ears only**. A mean that has seen the val or test
  subjects is not a prior; it is a leak.
- The error is measured on **val or test**, never on train. Scoring the mean predictor on
  the very ears it was fitted to is in-sample: the mean is the least-squares optimum for
  exactly that set, so the error is minimised by construction. It would make the baseline
  look better than it is and therefore make the model's improvement look smaller.

For a mean estimator the expected in-sample and out-of-sample MSEs are
`sigma^2 (1 - 1/n)` and `sigma^2 (1 + 1/n)`, so at n = 348 training ears (CARLab) the two
differ by roughly 0.3% of the RMSE, and at n = 116 (HUTUBS) by roughly 0.9%. Small — but
smallness is not the reason to pick one. The model's 1.372 dB is an out-of-sample number,
so its comparator has to be one too, or "72.8% reduction" is not a ratio of comparable
quantities. This notebook prints the in-sample figure as a labelled control so the size of
the effect is visible rather than argued about.

Right ears are mirrored across the median plane before averaging, exactly as in
`MonauralHRTFDataset`, so the mean is taken in a single left-ear frame.

For contrast this notebook also reports two variants it does **not** use: the arithmetic
mean of per-ear RMSEs, and a leaky baseline built from the mean of *all* subjects. Seeing
the three side by side is the quickest way to check that the right one is being quoted.

In [ ]:
# ==========================================================================
# Cell 0 — mount Drive and unpack the archives
#
# Same layout as the training notebook, so the two agree by construction:
#   CARLab  : ELEC5021/CARLab/CARLab_44k1_left.zip + _right.zip -> CARLab_44k1_left/right
#   others  : ELEC5020/<db>.zip (one file)                      -> npy_left / npy_right
# See docs/DATA.md for the Drive links behind these paths.
# ==========================================================================
import os, zipfile
from google.colab import drive

drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
BASE  = '/content/dataset'

DATASETS_SPEC = [
    dict(id=0, tag='CARLab',
         zips=[f'{DRIVE}/ELEC5021/CARLab/CARLab_44k1_left.zip',
               f'{DRIVE}/ELEC5021/CARLab/CARLab_44k1_right.zip'],
         extract_to=f'{BASE}/carlab',
         dirs=('CARLab_44k1_left', 'CARLab_44k1_right')),
    dict(id=1, tag='HUTUBS',
         zips=[f'{DRIVE}/ELEC5020/HUTUBS.zip'],
         extract_to=f'{BASE}/hutubs',
         dirs=('npy_left', 'npy_right')),
    dict(id=2, tag='SONICOM',
         zips=[f'{DRIVE}/ELEC5020/SONICOM.zip'],
         extract_to=f'{BASE}/sonicom',
         dirs=('npy_left', 'npy_right')),
]

KNOWN_SUFFIX = ['_left_full.npy', '_left.npy', '_Left.npy', '_L.npy', '-left.npy']
L2R = [('_left_full', '_right_full'), ('_left', '_right'), ('_Left', '_Right'),
       ('-left', '-right'), ('_L.', '_R.')]


def find_dir(root, target):
    """Recursively find a folder named target, skipping __MACOSX; take the one with the most .npy"""
    cands = []
    for dp, _, fns in os.walk(root):
        if '__MACOSX' in dp:
            continue
        if os.path.basename(dp) == target:
            cands.append((sum(f.endswith('.npy') for f in fns), dp))
    cands.sort(reverse=True)
    return cands[0][1] if cands else None


def detect_suffix(d):
    """Detect the suffix in '<subject_id><suffix>'; fall back to the longest common suffix."""
    names = sorted(f for f in os.listdir(d) if f.endswith('.npy'))
    assert names, f'no .npy in {d}'
    for s in KNOWN_SUFFIX:
        if all(n.endswith(s) for n in names):
            return s
    s = names[0]
    for n in names[1:]:
        i = 0
        while i < min(len(s), len(n)) and s[-1 - i] == n[-1 - i]:
            i += 1
        s = s[len(s) - i:]
    return s


def right_of(sl):
    for a, b in L2R:
        if a in sl:
            return sl.replace(a, b)
    return None


DATASETS = []
for spec in DATASETS_SPEC:
    tag, out = spec['tag'], spec['extract_to']
    os.makedirs(out, exist_ok=True)
    missing = [z for z in spec['zips'] if not os.path.exists(z)]
    if missing:
        print(f'[{tag}] * skipped, archive not found: {missing}')
        continue
    for zp in spec['zips']:
        with zipfile.ZipFile(zp) as z:
            z.extractall(out)

    dl, dr = spec['dirs']
    ld, rd = find_dir(out, dl), find_dir(out, dr)
    if ld is None or rd is None:
        print(f'[{tag}] * {dl} / {dr} not found after extraction — check the zip layout')
        continue

    sl = detect_suffix(ld)
    sr = right_of(sl) or detect_suffix(rd)
    DATASETS.append(dict(id=spec['id'], tag=tag, left=ld, right=rd, suffix=(sl, sr)))
    n = len([f for f in os.listdir(ld) if f.endswith('.npy')])
    print(f'[{tag}] {n} left / '
          f'{len([f for f in os.listdir(rd) if f.endswith(".npy")])} right  '
          f'suffix {sl!r} / {sr!r}')

assert DATASETS, 'no database loaded'

In [ ]:
# ==========================================================================
# Cell 1 — constants and integrity check
#
# These must match the training configuration exactly. num_bins is 93, not 129:
# the raw archives hold 129 bins to 22.05 kHz, and training keeps 0..92, i.e.
# DC to 15.85 kHz, because CARLab is digitally filtered at 16.31 kHz.
# ==========================================================================
import numpy as np
import healpy as hp

FS, NFFT, RAW_BINS, NPIX, NSIDE = 44100, 256, 129, 3072, 16
N_BINS      = 93                      # bins kept for training and for this baseline
BAND_LO_HZ  = 0                       # config['band_lo_hz'] — the loss covers everything
DF          = FS / NFFT               # 172.266 Hz
FREQS       = np.arange(N_BINS) * DF
BAND        = FREQS >= BAND_LO_HZ     # all True at band_lo_hz = 0; kept explicit on purpose

print(f'fs {FS} | df {DF:.3f} Hz | keeping {N_BINS}/{RAW_BINS} bins '
      f'-> DC to {FREQS[-1]/1000:.2f} kHz | band mask keeps {BAND.sum()}/{N_BINS}')

# ---- mirror permutation: right ears are folded into the left-ear frame ----
MIRROR_SRC = next((p for p in (
    f'{DRIVE}/ELEC5021/analysis/healpix_mirror_idx.npy',   # ← 现在的位置
    f'{DRIVE}/HRTF_Checkpoints/healpix_mirror_idx.npy',    # healpix_mirror.py 的旧输出
    f'{DRIVE}/healpix_mirror_idx.npy')                     # 更早的根目录版
    if os.path.exists(p)), f'{DRIVE}/ELEC5021/analysis/healpix_mirror_idx.npy')
if os.path.exists(MIRROR_SRC):
    MIRROR = np.load(MIRROR_SRC)
    print(f'mirror index loaded from Drive')
else:
    th, ph = hp.pix2ang(NSIDE, np.arange(NPIX))
    MIRROR = hp.ang2pix(NSIDE, th, (-ph) % (2 * np.pi))
    print('mirror index rebuilt with healpy')
assert len(np.unique(MIRROR)) == NPIX, 'mirror is not a bijection'
assert np.array_equal(MIRROR[MIRROR], np.arange(NPIX)), 'mirror is not an involution'
print('mirror permutation OK (exact bijection and involution)')

# ---- shape check on one file per database ----
for d in DATASETS:
    for side in ('left', 'right'):
        f = sorted(x for x in os.listdir(d[side]) if x.endswith('.npy'))[0]
        x = np.load(os.path.join(d[side], f))
        assert x.shape == (RAW_BINS, NPIX), f"{d['tag']}-{side}: shape {x.shape}"
    print(f"  {d['tag']:<9} shape ({RAW_BINS}, {NPIX}) OK")

In [ ]:
# ==========================================================================
# Cell 2 — load the frozen split and verify it still describes this data
#
# The hash is the same md5 of the sorted subject list that build_split() stores.
# If it no longer matches, the archives on Drive are not the ones the model was
# trained on, and every number below would be for a different partition.
# ==========================================================================
import json, hashlib

split_path = f'{DRIVE}/HRTF_Checkpoints/split_v21.json'
assert os.path.exists(split_path), (
    f'{split_path} not found. The same file is committed as analysis/split_v21.json '
    'in the repository, and a copy is embedded in every .pth under the key "split".')
SPLIT = json.load(open(split_path))


def subject_ids(d):
    sfx = d['suffix'][0]
    return sorted({f[:-len(sfx)] for f in os.listdir(d['left']) if f.endswith(sfx)})


print(f"{'DB':<9}{'total':>7}{'train':>7}{'val':>6}{'test':>6}   hash")
for d in DATASETS:
    tag = d['tag']
    assert tag in SPLIT, f'{tag} missing from split_v21.json'
    ids = subject_ids(d)
    h = hashlib.md5(','.join(ids).encode()).hexdigest()[:12]
    stored = SPLIT[tag]['hash']
    ok = (h == stored)
    s = SPLIT[tag]
    print(f"{tag:<9}{len(ids):>7}{len(s['train']):>7}{len(s['val']):>6}{len(s['test']):>6}"
          f"   {h} {'OK' if ok else f'MISMATCH vs stored {stored}'}")
    assert ok, (f'{tag}: subject list has changed since the split was frozen '
                f'(stored {stored}, now {h}). Stop — the baselines would not correspond '
                'to the partitions the model was scored on.')

print('\nall subject-list hashes match the frozen split')

In [ ]:
# ==========================================================================
# Cell 3 — compute the baselines
#
# Streaming: the training mean is accumulated one ear at a time, so nothing
# larger than the val+test slice of a single database is ever held in memory.
#
# Three quantities per partition. Only the first is the one to quote:
#   pooled     sqrt( mean over (ears x bins x directions) of (x - mu_train)^2 )
#              -> identical convention to evaluate(), comparable with 1.372 dB
#   arithmetic mean over ears of each ear's own RMSE
#              -> systematically SMALLER; different quantity, not interchangeable
#   leaky      pooled, but against the mean of ALL subjects including val/test
#              -> reported only to show why the training mean is the right one
# ==========================================================================
import numpy as np

def load_ear(d, sid, side):
    """One ear, cropped to N_BINS, right ears mirrored into the left-ear frame."""
    sfx = d['suffix'][0] if side == 'left' else d['suffix'][1]
    x = np.load(os.path.join(d[side], f'{sid}{sfx}')).astype(np.float32)[:N_BINS]
    return x[:, MIRROR] if side == 'right' else x


results = {}
for d in DATASETS:
    tag = d['tag']
    s = SPLIT[tag]

    # ---- pass over train: accumulate the mean, one ear at a time ----
    sum_tr = np.zeros((N_BINS, NPIX), dtype=np.float64)
    sumsq_tr, n_tr = 0.0, 0
    for sid in s['train']:
        for side in ('left', 'right'):
            x = load_ear(d, sid, side)
            sum_tr += x
            sumsq_tr += float((x.astype(np.float64)[BAND, :] ** 2).sum())
            n_tr += 1
    mu_tr = (sum_tr / n_tr).astype(np.float32)

    # In-sample error of the same predictor, for reference only. No second pass is
    # needed: sum_i ||x_i - mu||^2 = sum_i ||x_i||^2 - n ||mu||^2, exactly, because
    # sum_i x_i = n mu. This is what "scoring the baseline on train" would give.
    sse_train = sumsq_tr - n_tr * float((mu_tr.astype(np.float64)[BAND, :] ** 2).sum())
    cnt_train = n_tr * int(BAND.sum()) * NPIX

    # ---- hold val and test in memory (small) ----
    held = {}
    for part in ('val', 'test'):
        held[part] = [load_ear(d, sid, side)
                      for sid in s[part] for side in ('left', 'right')]

    # ---- all-subject mean, for the leaky variant only ----
    sum_all = sum_tr.copy()
    n_all = n_tr
    for part in ('val', 'test'):
        for x in held[part]:
            sum_all += x
            n_all += 1
    mu_all = (sum_all / n_all).astype(np.float32)

    res = {'n_train_ears': n_tr,
           'train_in_sample': float(np.sqrt(max(sse_train, 0.0) / cnt_train))}
    for part in ('val', 'test'):
        ears = held[part]
        if not ears:
            continue
        sse = sse_leaky = 0.0
        cnt = 0
        per_ear = []
        for x in ears:
            e = (x - mu_tr)[BAND, :]
            sse += float((e.astype(np.float64) ** 2).sum())
            cnt += e.size
            per_ear.append(float(np.sqrt((e.astype(np.float64) ** 2).mean())))
            el = (x - mu_all)[BAND, :]
            sse_leaky += float((el.astype(np.float64) ** 2).sum())
        per_ear = np.array(per_ear)
        res[part] = dict(
            n_subjects=len(s[part]), n_ears=len(ears),
            pooled=float(np.sqrt(sse / cnt)),
            arithmetic=float(per_ear.mean()),
            std_across_ears=float(per_ear.std()),
            leaky_all_subject_mean=float(np.sqrt(sse_leaky / cnt)),
        )
    results[tag] = res
    del held, sum_tr, sum_all
    print(f'{tag:<9} done  (train mean over {n_tr} ears)')

In [ ]:
# ==========================================================================
# Cell 4 — the table, and a check against the numbers already on record
# ==========================================================================
MODEL = {                      # from the locked run; here only for the ratio columns
    'CARLab':  dict(val=1.372, test=1.2909),
    'HUTUBS':  dict(val=1.813, test=1.7680),
    'SONICOM': dict(val=1.454, test=1.4295),
}
KNOWN_VAL_BASELINE = {         # printed by Cell 3 of the training notebook
    'CARLab': 5.053, 'HUTUBS': 5.162, 'SONICOM': 4.977,
}

for part in ('val', 'test'):
    print(f'\n{part.upper()} — mean-HRTF baseline, {BAND_LO_HZ}-{FREQS[-1]:.0f} Hz, '
          f"each database's own training mean")
    print(f"  {'DB':<9}{'subj':>6}{'ears':>6}{'baseline':>10}{'model':>8}"
          f"{'reduction':>11}{'arith':>9}{'leaky':>9}")
    for tag, r in results.items():
        if part not in r:
            continue
        p = r[part]
        m = MODEL.get(tag, {}).get(part)
        red = f'{(1 - m / p["pooled"]) * 100:>10.1f}%' if m else f'{"":>11}'
        mm = f'{m:>8.3f}' if m else f'{"":>8}'
        print(f"  {tag:<9}{p['n_subjects']:>6}{p['n_ears']:>6}{p['pooled']:>10.3f}"
              f"{mm}{red}{p['arithmetic']:>9.3f}{p['leaky_all_subject_mean']:>9.3f}")

print('\n' + '=' * 78)
print('CHECK — val baselines against the values already printed by the training notebook')
worst = 0.0
for tag, ref in KNOWN_VAL_BASELINE.items():
    if tag in results and 'val' in results[tag]:
        got = results[tag]['val']['pooled']
        worst = max(worst, abs(got - ref))
        print(f'  {tag:<9} recomputed {got:.4f}  vs on record {ref:.3f}  '
              f'delta {got - ref:+.4f}  {"OK" if abs(got - ref) < 0.005 else "*** MISMATCH"}')
print(f'  largest deviation {worst:.4f} dB')
if worst >= 0.005:
    print('  *** Something differs from the training configuration — check num_bins, the '
          'mirror, band_lo_hz, or whether the archives are the same ones.')
else:
    print('  Convention reproduced exactly. The test column above is therefore trustworthy.')

print('\n' + '=' * 78)
print('IN-SAMPLE CONTROL — the same mean predictor scored on the training ears it was')
print('built from. Not a baseline: the mean is the least-squares optimum for exactly those')
print('ears, so this is minimised by construction. Shown to size the effect.')
print(f"  {'DB':<9}{'train ears':>11}{'in-sample':>11}{'val':>9}{'gap':>9}")
for tag, r in results.items():
    v = r.get('val', {}).get('pooled')
    ti = r['train_in_sample']
    print(f"  {tag:<9}{r['n_train_ears']:>11}{ti:>11.3f}{v:>9.3f}{v - ti:>+9.3f}")
print('  The gap is small — for a mean estimator the expected ratio is')
print('  sqrt((1-1/n)/(1+1/n)), a fraction of a percent at these n. Small is not the')
print('  point: the model figure it is compared against is out-of-sample, so the')
print('  baseline must be too, or the reduction percentage is not a ratio of like things.')

print('\nNotes on the last two columns of the tables above:')
print('  arith  arithmetic mean of per-ear RMSE. Always <= pooled, because pooled is the')
print('         quadratic mean. Do not mix the two in one table.')
print('  leaky  baseline built from the mean of ALL subjects, val and test included.')
print('         Shown only as a control: that mean has seen the subjects it is scored on,')
print('         so it is not a prior and must not be quoted.')

In [ ]:
# ==========================================================================
# Cell 5 — write the results out
#
# Two files next to the checkpoints on Drive: a JSON for any later script, and a
# markdown block ready to paste into README.md and docs/PROJECT_LOG.md.
# ==========================================================================
import json, time

OUT_DIR = f'{DRIVE}/HRTF_Checkpoints'
stamp = time.strftime('%Y-%m-%d %H:%M:%S')

payload = {
    'generated': stamp,
    'convention': ('pooled RMSE over (ears x bins in band x directions) against the '
                   'training-set mean spectrum of the same database; right ears mirrored '
                   'into the left-ear frame'),
    'num_bins': N_BINS, 'sample_rate': FS, 'band_lo_hz': BAND_LO_HZ,
    'band_hi_hz': float(FREQS[-1]), 'split_file': 'split_v21.json',
    'baselines': results,
}
with open(f'{OUT_DIR}/baselines.json', 'w') as fh:
    json.dump(payload, fh, indent=1)

lines = [
    f'<!-- generated by notebooks/Compute_Baselines.ipynb, {stamp} -->',
    '',
    '| Database | val baseline | test baseline | val subjects | test subjects |',
    '|---|---:|---:|---:|---:|',
]
for tag, r in results.items():
    v = r.get('val', {}).get('pooled')
    t = r.get('test', {}).get('pooled')
    lines.append(f"| {tag} | {v:.3f} dB | {t:.3f} dB | "
                 f"{r['val']['n_subjects']} | {r['test']['n_subjects']} |")
md = '\n'.join(lines)
with open(f'{OUT_DIR}/baselines.md', 'w') as fh:
    fh.write(md + '\n')

print(f'written to {OUT_DIR}/baselines.json and baselines.md\n')
print(md)
print('\nCopy baselines.md into the repo as results/baselines.md, and fill the same '
      'numbers into the Baseline column of README.md.')

## Where these numbers go

1. `results/baselines.md` in the repository — commit the generated block.
2. The **Baseline** column of the results table in `README.md`.
3. `docs/PROJECT_LOG.md`, as a dated entry recording that the test baselines now exist
   and by what convention.

Quote the **pooled** column, from the **val** or **test** row that matches the model
figure beside it. The in-sample control is a diagnostic, not a number for any table. If a
per-subject spread is wanted, quote `std_across_ears` from `baselines.json` rather than
swapping in the arithmetic mean — the log already records one confusion between those two
conventions and does not need a second.